In [1]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime

In [3]:
url = "https://www.ncei.noaa.gov/data/local-climatological-data/access/2021/"
target_date = "2024-01-19 10:27" 
def main():
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    for row in soup.find_all('tr')[1:]:  # Skip header
        cols = row.find_all('td')
        if len(cols) > 1:
            filename = cols[0].find('a')['href']
            file_url = url + filename
            last_mod = cols[1].text.strip()
    
            # Parse date (format: "2021-07-15 14:23")
            file_date = last_mod.split(' ')[0]
            if file_date == target_date:
                print(f"Downloading {filename}...")
                file_data = requests.get(file_url)
                with open(filename, 'wb') as f:
                    f.write(file_data.content)
if __name__ == "__main__":
    main()

In [4]:
import requests
from bs4 import BeautifulSoup

def main():
    url = "https://www.ncei.noaa.gov/data/local-climatological-data/access/2021/"
    target_date = "2024-01-19 10:27"  

    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    for row in soup.find_all('tr')[1:]:
        cols = row.find_all('td')
        if len(cols) > 1:
            filename = cols[0].find('a')['href']
            file_url = url + filename
            file_date = cols[1].text.strip().split(' ')[0]
            
            if file_date == target_date:
                print(f"Downloading {filename}...")
                file_data = requests.get(file_url)
                with open(filename, 'wb') as f:
                    f.write(file_data.content)
                print("Download complete.")
                return

if __name__ == "__main__":
    main()   

In [8]:
import requests
from bs4 import BeautifulSoup

def extract_data():
    url = "https://www.ncei.noaa.gov/data/local-climatological-data/access/2021/"
    target_datetime = "2024-01-19 10:27"    # Full date and time (adjust as needed)

    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    for row in soup.find_all('tr')[1:]:
        cols = row.find_all('td')
        if len(cols) > 1:
            filename = cols[0].find('a')['href']
            if not filename.endswith('.csv'):
                continue
            file_url = url + filename
            last_modified = cols[1].text.strip()  # Includes date and time
            
            if last_modified == target_datetime:
                print(f"Downloading {filename} as data.csv...")
                file_data = requests.get(file_url)
                with open('data.csv', 'wb') as f:
                    f.write(file_data.content)
                print("Download complete.")
                return
extract_data()

In [10]:
import requests
import csv
from bs4 import BeautifulSoup

# Step 1: Fetch the web page
url = "https://www.ncei.noaa.gov/data/local-climatological-data/access/2021/"
response = requests.get(url)
response.raise_for_status()  # Check for request errors

# Step 2: Parse HTML
soup = BeautifulSoup(response.content, 'html.parser')

# Step 3: Find the table (example: first table with class 'wikitable')
table = soup.find('table', {'class': 'wikitable'})

# Step 4: Extract headers
headers = [th.get_text(strip=True) for th in table.find_all('th')]
rows = []

# Step 5: Extract data rows
for tr in table.find_all('tr')[1:]:  # Skip header row
    cells = tr.find_all(['td', 'th'])
    row = [cell.get_text(strip=True) for cell in cells]
    if row:
        rows.append(row)

# Step 6: Save to CSV
with open('scraped_data.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(headers)  # Write header
    writer.writerows(rows)    # Write data rows

print("Data successfully saved to scraped_data.csv")   

AttributeError: 'NoneType' object has no attribute 'find_all'

In [11]:
import requests
import csv
from bs4 import BeautifulSoup

# Step 1: Fetch the web page
url = "https://www.ncei.noaa.gov/data/local-climatological-data/access/2021/"
response = requests.get(url)
response.raise_for_status()  # Check for request errors

# Step 2: Parse HTML
soup = BeautifulSoup(response.content, 'html.parser')

# Step 3: Find the table - the website likely doesn't have a table with class 'wikitable'
# Instead, let's find the appropriate table or elements that contain the data
# First, check if any table exists
table = soup.find('table')

# If no table is found, we need to handle this case
if table is None:
    print("No table found on the page. Checking for links instead...")
    # This website likely lists files as links, so let's extract those
    links = soup.find_all('a')
    
    # Extract link text and href
    headers = ["Filename", "URL"]
    rows = []
    for link in links:
        # Skip parent directory link
        if link.text.strip() and link.text != "Parent Directory":
            filename = link.text.strip()
            href = link.get('href', '')
            rows.append([filename, href])
    
    # Save to CSV
    with open('scraped_data.csv', 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(headers)  # Write header
        writer.writerows(rows)    # Write data rows
    
    print("Links successfully saved to scraped_data.csv")
else:
    # Original table processing code if a table is found
    headers = [th.get_text(strip=True) for th in table.find_all('th')]
    rows = []
    
    for tr in table.find_all('tr')[1:]:  # Skip header row
        cells = tr.find_all(['td', 'th'])
        row = [cell.get_text(strip=True) for cell in cells]
        if row:
            rows.append(row)
    
    # Save to CSV
    with open('scraped_data.csv', 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(headers)  # Write header
        writer.writerows(rows)    # Write data rows
    
    print("Table data successfully saved to scraped_data.csv")

Table data successfully saved to scraped_data.csv


In [12]:
import pandas as pd

In [17]:
import os
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import urllib.parse

def download_csv_by_datetime(base_url, target_datetime_str, directory="downloadeds", file_extension=".csv"):
    # Parse target datetime
    target_dt = datetime.strptime(target_datetime_str, "%Y-%m-%d %H:%M")

    # Create directory if it doesn't exist
    os.makedirs(directory, exist_ok=True)

    # Step 1: Fetch and parse webpage
    try:
        response = requests.get(base_url)
        response.raise_for_status()
    except requests.RequestException as e:
        print(f"Error fetching webpage: {e}")
        return None

    soup = BeautifulSoup(response.content, 'html.parser')

    # Find all CSV links
    csv_links = []
    for link in soup.find_all('a', href=True):
        href = link['href']
        if href.endswith(file_extension):
            full_url = urllib.parse.urljoin(base_url, href)
            csv_links.append(full_url)

    print(f"Found {len(csv_links)} CSV file(s). Checking Last-Modified headers...")

    for csv_url in csv_links:
        try:
            # Use HEAD request to get Last-Modified header
            head_response = requests.head(csv_url, allow_redirects=True)
            head_response.raise_for_status()

            last_modified_str = head_response.headers.get('Last-Modified')
            if not last_modified_str:
                print(f"No Last-Modified header: {csv_url}")
                continue

            # Parse Last-Modified (e.g., 'Wed, 18 Oct 2024 12:34:56 GMT')
            last_modified_dt = datetime.strptime(last_modified_str, "%a, %d %b %Y %H:%M")

            # Compare full datetime
            if last_modified_dt == target_dt:
                print(f"Match found! Downloading: {csv_url}")

                # Download file
                download_response = requests.get(csv_url)
                download_response.raise_for_status()

                filename = os.path.join(directory, csv_url.split('/')[-1])
                with open(filename, 'wb') as f:
                    f.write(download_response.content)

                print(f"Downloaded and saved to: {filename}")
                return filename

        except requests.RequestException as e:
            print(f"Error accessing {csv_url}: {e}")
            continue

    print("No matching CSV file found with the specified date and time.")
    return None

# Example usage
base_url = "https://www.ncei.noaa.gov/data/local-climatological-data/access/2021/"  # Replace with actual URL
target_datetime = "2024-01-19 14:51"
download_csv_by_datetime(base_url, target_datetime)   

Found 13539 CSV file(s). Checking Last-Modified headers...
Match found! Downloading: https://www.ncei.noaa.gov/data/local-climatological-data/access/2021/02456099999.csv
Downloaded and saved to: downloaded_csvs/02456099999.csv


'downloaded_csvs/02456099999.csv'

In [31]:
import os
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
import urllib.parse

def download_csv_if_time_matches(url, target_dt, session, directory):
    try:
        # Check Last-Modified header
        head_resp = session.head(url, timeout=10)
        head_resp.raise_for_status()
        last_mod_str = head_resp.headers.get('Last-Modified')
        if not last_mod_str:
            return None

        # Parse time (format: 'Wed, 15 Oct 2025 10:30:00 GMT')
        last_mod_dt = datetime.strptime(last_mod_str, "%a, %d %b %Y %H:%M:%S %Z")
        if last_mod_dt.replace(second=0, microsecond=0) == target_dt:
            # Download if matches
            get_resp = session.get(url, timeout=10)
            get_resp.raise_for_status()
            filename = os.path.join(directory, url.split('/')[-1])
            with open(filename, 'wb') as f:
                f.write(get_resp.content)
            print(f"Downloaded: {filename}")
            return filename
    except Exception as e:
        print(f"Error: {url} - {e}")
    return None

def download_csvs_by_timestamp(base_url, target_datetime_str, directory="downloads", max_workers=10):
    target_dt = datetime.strptime(target_datetime_str, "%Y-%m-%d %H:%M")
    os.makedirs(directory, exist_ok=True)

    with requests.Session() as session:
        # Fetch page
        resp = session.get(base_url)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.content, 'html.parser')

        # Extract CSV links
        csv_urls = [
            urllib.parse.urljoin(base_url, a['href'])
            for a in soup.find_all('a', href=True)
            if a['href'].endswith('.csv')
        ]

        # Parallel download check
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [
                executor.submit(download_csv_if_time_matches, url, target_dt, session, directory)
                for url in csv_urls
            ]
            return [f.result() for f in futures if f.result()]

# Example usage
base_url = "https://www.ncei.noaa.gov/data/local-climatological-data/access/2021/"
target_time = "2024-01-19 15:45"  # Adjust to desired date/time
download_csvs_by_timestamp(base_url, target_time)   

Downloaded: downloads/01002099999.csv
Downloaded: downloads/01368099999.csv


KeyboardInterrupt: 

Downloaded: downloads/03761099999.csvError: https://www.ncei.noaa.gov/data/local-climatological-data/access/2021/03761099999.csv - Socket operation on non-socket
Downloaded: downloads/03955099999.csv
Downloaded: downloads/03808099999.csv
Downloaded: downloads/04203099999.csvError: https://www.ncei.noaa.gov/data/local-climatological-data/access/2021/04203099999.csv - Socket operation on non-socket
Downloaded: downloads/06449099999.csv
Downloaded: downloads/06639099999.csv
Downloaded: downloads/08501099999.csv
Downloaded: downloads/08548099999.csv
Downloaded: downloads/08926099999.csv
Downloaded: downloads/15108099999.csv
Downloaded: downloads/15297099999.csv
Downloaded: downloads/16643099999.csv
Downloaded: downloads/24361099999.csv
Downloaded: downloads/28517099999.csv
Downloaded: downloads/30393099999.csv
Downloaded: downloads/30309099999.csv
Downloaded: downloads/31168099999.csv
Downloaded: downloads/31253099999.csv
Downloaded: downloads/33429099999.csv
Downloaded: downloads/40373099

In [38]:
import os
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import urllib.parse
from typing import Optional, List
import pandas as pd

def download_csv_if_time_matches(url: str, target_dt: datetime, session: requests.Session, directory: str) -> Optional[str]:

    try:
        # Check Last-Modified header
        head_resp = session.head(url, timeout=10)
        head_resp.raise_for_status()
        last_mod_str = head_resp.headers.get('Last-Modified')
        if not last_mod_str:
            return None

        # Parse time (format: 'Wed, 15 Oct 2025 10:30:00 GMT')
        last_mod_dt = datetime.strptime(last_mod_str, "%a, %d %b %Y %H:%M:%S %Z")
        if last_mod_dt.replace(second=0, microsecond=0) == target_dt:
            # Download if matches
            get_resp = session.get(url, timeout=10)
            get_resp.raise_for_status()
            filename = os.path.join(directory, url.split('/')[-1])
            with open(filename, 'wb') as f:
                f.write(get_resp.content)
            print(f"Downloaded: {filename}")
            return filename
    except Exception as e:
        print(f"Error: {url} - {e}")
    return None

def download_first_csv_by_timestamp(base_url: str, target_datetime_str: str, directory: str = "downloads") -> Optional[str]:

    target_dt = datetime.strptime(target_datetime_str, "%Y-%m-%d %H:%M")
    os.makedirs(directory, exist_ok=True)

    with requests.Session() as session:
        # Fetch page
        resp = session.get(base_url)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.content, 'html.parser')

        # Extract CSV links
        csv_urls = [
            urllib.parse.urljoin(base_url, a['href'])
            for a in soup.find_all('a', href=True)
            if a['href'].endswith('.csv')
        ]

        # Check each URL sequentially and return after first match
        for url in csv_urls:
            result = download_csv_if_time_matches(url, target_dt, session, directory)
            if result:
                return result
                
        print("No CSV files found matching the specified timestamp.")
        return None

# Example usage
base_url = "https://www.ncei.noaa.gov/data/local-climatological-data/access/2021/"
target_time = "2024-01-19 15:45"  
result = download_first_csv_by_timestamp(base_url, target_time)


Downloaded: downloads/01002099999.csv
Successfully downloaded: downloads/01002099999.csv


In [41]:
df = pd.read_csv(result)

In [34]:
df.head()

,STATION,DATE,LATITUDE,LONGITUDE,ELEVATION,NAME,REPORT_TYPE,SOURCE,HourlyAltimeterSetting,HourlyDewPointTemperature,...,BackupDirection,BackupDistance,BackupDistanceUnit,BackupElements,BackupElevation,BackupEquipment,BackupLatitude,BackupLongitude,BackupName,WindEquipmentChangeDate
0,1002099999,2021-01-01T01:00:00,80.05,16.25,8.0,"VERLEGENHUKEN, NO",FM-12,4,NaN,22.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1002099999,2021-01-01T22:00:00,80.05,16.25,8.0,"VERLEGENHUKEN, NO",FM-12,4,NaN,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1002099999,2021-01-02T00:00:00,80.05,16.25,8.0,"VERLEGENHUKEN, NO",FM-12,4,NaN,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1002099999,2021-01-02T23:00:00,80.05,16.25,8.0,"VERLEGENHUKEN, NO",FM-12,4,NaN,20.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1002099999,2021-01-04T16:00:00,80.05,16.25,8.0,"VERLEGENHUKEN, NO",FM-12,4,NaN,21.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
# HourlyDryBulbTemperature
df['HourlyDryBulbTemperature'].max()

np.float64(49.0)

In [49]:
import os
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import urllib.parse

def main(base_url: str, target_datetime_str: str, directory: str = "downloads") -> str:
    target_dt = datetime.strptime(target_datetime_str, "%Y-%m-%d %H:%M")
    os.makedirs(directory, exist_ok=True)

    with requests.Session() as session:
        # Fetch and parse page
        resp = session.get(base_url)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.content, 'html.parser')

        # Extract CSV links
        csv_urls = [
            urllib.parse.urljoin(base_url, a['href'])
            for a in soup.find_all('a', href=True)
            if a['href'].endswith('.csv')
        ]

        # Check each file's Last-Modified header
        for url in csv_urls:
            try:
                head_resp = session.head(url, timeout=10)
                head_resp.raise_for_status()
                last_mod_str = head_resp.headers.get('Last-Modified')
                if not last_mod_str:
                    continue

                last_mod_dt = datetime.strptime(last_mod_str, "%a, %d %b %Y %H:%M:%S %Z")
                if last_mod_dt.replace(second=0, microsecond=0) == target_dt:
                    # Download on match
                    get_resp = session.get(url, timeout=10)
                    get_resp.raise_for_status()
                    filename = os.path.join(directory, url.split('/')[-1])
                    with open(filename, 'wb') as f:
                        f.write(get_resp.content)
                    # print(f"Downloaded: {filename}")
                    return filename
            except Exception as e:
                print(f"Error: {url} - {e}")

        print("No file found with the specified timestamp.")
        return None

if __name__ == "__main__":
    result1 = main(
        "https://www.ncei.noaa.gov/data/local-climatological-data/access/2021/",
        "2024-01-19 15:45")   
    df = pd.read_csv(result1)
    print(df['HourlyDryBulbTemperature'].max())

49.0


In [53]:
!pip show urllib